# TRACERx evolution sensitivities

Adapted from the submitted repository, commit `fce9ade29ef6de731407a7fcf14098f30a272cd3`, for the public-data workflow. Current definitions and figure mappings are documented in `docs/`.


In [ ]:
import os
from pathlib import Path
from IPython.display import display
_candidates = [Path.cwd(), *Path.cwd().parents]
_project = next((p for p in _candidates if (p / "src/mitoxy/data.py").is_file()), None)
if _project is None:
    raise RuntimeError("Launch this notebook within the Mitoxy repository")
os.environ["MITOXY_PROJECT_ROOT"] = str(_project)


# Supplementary Figure S4 — TRACERx regional variation and evolutionary omnibus tests

Panel A quantifies regional variation gene by gene. Panel B fits adjusted
multi-level models for the complete evolutionary classifications. These global
tests are deliberately distinct from the focused punctuated-versus-other
contrast in the main text.


In [ ]:
from pathlib import Path
import hashlib
import os
import re
import warnings

import numpy as np
import pandas as pd
import pyreadr
import statsmodels.api as sm
from scipy.stats import pearsonr, spearmanr
from statsmodels.stats.multitest import multipletests

PROJECT_ROOT = Path(
    os.environ["MITOXY_PROJECT_ROOT"]
).resolve()
MANIFEST_PATH = (
    PROJECT_ROOT / "data" / "provenance" /
    "paper_public_source_manifest.csv"
)
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "figure_4_clean"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GENES = ["BAK1", "BAX", "BID", "SLC7A11"]
manifest = pd.read_csv(MANIFEST_PATH)


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def source_path(dataset, role):
    match = manifest.loc[
        manifest["dataset"].eq(dataset)
        & manifest["role"].eq(role)
    ]
    if len(match) != 1:
        raise RuntimeError(
            f"Expected one source for ({dataset!r}, {role!r}); "
            f"found {len(match)}"
        )
    row = match.iloc[0]
    path = PROJECT_ROOT / str(row["relative_path"])
    if not path.is_file():
        raise FileNotFoundError(path)
    if path.stat().st_size != int(row["size_bytes"]):
        raise RuntimeError(f"Source-size mismatch: {path}")
    if sha256_file(path) != str(row["sha256"]):
        raise RuntimeError(f"Source SHA-256 mismatch: {path}")
    return path


RAW = {
    "tpm": source_path("TRACERx Renal", "regional tumor TPM"),
    "annotation": source_path("TRACERx Renal", "regional annotation"),
    "clinical": source_path("TRACERx Renal", "clinical annotation"),
}

display(pd.DataFrame({"analysis_key": RAW.keys(), "path": RAW.values()}))
print("Public-source integrity audit: PASS")


In [ ]:
def zscore(values):
    values = pd.Series(values, index=getattr(values, "index", None), dtype=float)
    standard_deviation = values.std(ddof=0)
    if not np.isfinite(standard_deviation) or standard_deviation == 0:
        raise ValueError("Cannot standardize a constant or invalid vector")
    return (values - values.mean()) / standard_deviation


def normalized(text):
    return re.sub(r"[^a-z0-9]+", " ", str(text).lower()).strip()


def find_column(columns, include, exclude=(), required=True):
    include = [normalized(term) for term in include]
    exclude = [normalized(term) for term in exclude]
    candidates = []
    for column in columns:
        name = normalized(column)
        if all(term in name for term in include) and not any(
            term in name for term in exclude
        ):
            candidates.append(column)
    if not candidates:
        if required:
            raise KeyError(
                f"No column matched include={include}, exclude={exclude}. "
                f"Available columns: {list(columns)}"
            )
        return None
    return sorted(candidates, key=lambda value: (len(str(value)), str(value)))[0]


def parse_grade(value):
    if pd.isna(value):
        return np.nan
    match = re.search(r"(?:G(?:RADE)?\s*)?([1-4])", str(value), flags=re.I)
    return float(match.group(1)) if match else np.nan


def parse_stage(value):
    if pd.isna(value):
        return np.nan
    text = str(value).upper().replace("STAGE", " ")
    match = re.search(r"\b(IV|III|II|I)\b", text)
    if match:
        return {"I": 1.0, "II": 2.0, "III": 3.0, "IV": 4.0}[
            match.group(1)
        ]
    match = re.search(r"\b([1-4])\b", text)
    return float(match.group(1)) if match else np.nan


def fit_hc3(data, outcome, predictors, focal):
    fields = [outcome, *predictors]
    complete = data[fields].replace([np.inf, -np.inf], np.nan).dropna()
    design = sm.add_constant(complete[predictors], has_constant="add")
    model = sm.OLS(
        complete[outcome].astype(float), design.astype(float)
    ).fit(cov_type="HC3", use_t=False)
    interval = model.conf_int().loc[focal]
    return {
        "n": int(model.nobs),
        "beta": float(model.params[focal]),
        "se_HC3": float(model.bse[focal]),
        "p_value": float(model.pvalues[focal]),
        "ci_low": float(interval.iloc[0]),
        "ci_high": float(interval.iloc[1]),
    }


def partial_spearman(data, x, y, covariates):
    fields = [x, y, *covariates]
    complete = data[fields].replace([np.inf, -np.inf], np.nan).dropna()
    ranked = complete.rank(method="average")
    design = sm.add_constant(ranked[covariates], has_constant="add")
    x_residual = sm.OLS(ranked[x], design).fit().resid
    y_residual = sm.OLS(ranked[y], design).fit().resid
    rho, p_value = pearsonr(x_residual, y_residual)
    return int(len(complete)), float(rho), float(p_value)


def dispersion_summary(values):
    values = np.asarray(values, dtype=float)
    differences = np.abs(values[:, None] - values[None, :])
    pairwise = differences[np.triu_indices(len(values), k=1)]
    return pd.Series({
        "n_regions": len(values),
        "Range": np.ptp(values),
        "SD": np.std(values, ddof=1),
        "MAD": np.median(np.abs(values - np.median(values))),
        "Mean pairwise |difference|": np.mean(pairwise),
        "Median pairwise |difference|": np.median(pairwise),
    })


In [ ]:
expression = pyreadr.read_r(str(RAW["tpm"]))[None]
expression.index = expression.index.astype(str).str.strip().str.upper()
expression.columns = expression.columns.astype(str).str.strip()
missing_genes = sorted(set(GENES) - set(expression.index))
if missing_genes:
    raise KeyError(f"TRACERx expression is missing {missing_genes}")

annotation = pd.read_csv(RAW["annotation"], sep="\t", dtype=str)
annotation.columns = [str(column).strip() for column in annotation.columns]

sample_column = find_column(annotation.columns, ["sample"])
patient_column = find_column(annotation.columns, ["patient"])
type_column = find_column(annotation.columns, ["type", "collapsed"])
condition_column = find_column(annotation.columns, ["condition"])
purity_column = find_column(annotation.columns, ["purity"])
wgi_column = find_column(annotation.columns, ["wgi"], required=False)
if wgi_column is None:
    wgi_column = find_column(annotation.columns, ["weighted", "genome"])
ith_column = find_column(annotation.columns, ["ith"], required=False)
if ith_column is None:
    ith_column = find_column(annotation.columns, ["intratum", "heterogeneity"])

annotation[sample_column] = annotation[sample_column].astype(str).str.strip()
annotation[patient_column] = annotation[patient_column].astype(str).str.strip()
primary_annotation = annotation.loc[
    annotation[type_column].astype(str).str.strip().str.upper().eq("PRIMARY")
].copy()

primary_samples = [
    sample for sample in primary_annotation[sample_column]
    if sample in expression.columns
]
if not primary_samples:
    raise RuntimeError("No PRIMARY annotation rows overlap expression samples")

# Figure 4 follows the TRACERx evolutionary analysis and standardizes the
# public TPM values directly across primary-tumor regions.
region_expression = (
    expression.loc[GENES, primary_samples]
    .T.apply(pd.to_numeric, errors="coerce")
)
if region_expression.isna().any().any():
    raise ValueError("Missing target-gene TPM values in primary regions")
region_gene_z = region_expression.apply(zscore, axis=0)
region_gene_z.columns = [f"z_{gene}" for gene in GENES]

regions = region_gene_z.copy()
regions["mitoxy"] = region_gene_z.mean(axis=1)
regions["sample_id"] = regions.index.astype(str)
regions = regions.merge(
    primary_annotation[[sample_column, patient_column]].drop_duplicates(),
    left_on="sample_id", right_on=sample_column,
    how="left", validate="one_to_one",
).rename(columns={patient_column: "patient"})

clinical = pd.read_csv(RAW["clinical"], sep="\t", header=1, dtype=str)
clinical.columns = [str(column).strip() for column in clinical.columns]
clinical_patient = find_column(clinical.columns, ["subject"])
clinical_grade = find_column(clinical.columns, ["fuhrman", "grade"])
clinical_stage = find_column(clinical.columns, ["overall", "stage"])
clinical_age = find_column(clinical.columns, ["age"])

clinical_table = pd.DataFrame({
    "patient": clinical[clinical_patient].astype(str).str.strip(),
    "grade": clinical[clinical_grade].map(parse_grade),
    "stage": clinical[clinical_stage].map(parse_stage),
    "age": pd.to_numeric(clinical[clinical_age], errors="coerce"),
}).drop_duplicates("patient")

patient_annotation = primary_annotation[
    [patient_column, condition_column]
].groupby(patient_column, as_index=False).first().rename(columns={
    patient_column: "patient", condition_column: "condition",
})

covariate_rows = primary_annotation[
    [patient_column, purity_column, wgi_column, ith_column]
].rename(columns={
    patient_column: "patient", purity_column: "purity",
    wgi_column: "wgi", ith_column: "ith",
})
for column in ["purity", "wgi", "ith"]:
    covariate_rows[column] = pd.to_numeric(
        covariate_rows[column], errors="coerce"
    )
covariates = covariate_rows.groupby("patient", as_index=False)[
    ["purity", "wgi", "ith"]
].mean()

patients = regions.groupby("patient", as_index=False).agg(
    mitoxy_raw=("mitoxy", "mean"),
    n_regions=("mitoxy", "size"),
)
patients["mitoxy"] = zscore(patients["mitoxy_raw"])
patients = (
    patients
    .merge(clinical_table, on="patient", how="left", validate="one_to_one")
    .merge(patient_annotation, on="patient", how="left", validate="one_to_one")
    .merge(covariates, on="patient", how="left", validate="one_to_one")
)
patients["punctuated"] = patients["condition"].astype(str).str.casefold().eq(
    "punctuated"
).astype(float)

print("Primary regions:", len(regions))
print("Patients represented:", patients["patient"].nunique())
display(patients[[
    "grade", "stage", "age", "purity", "wgi", "ith", "condition"
]].notna().sum().rename("n_complete").to_frame())


In [ ]:
OUTPUT_DIR=PROJECT_ROOT/"data"/"processed"/"supplementary_figure_s4"; OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
from statsmodels.regression.mixed_linear_model import MixedLM

def method_of_moments_variance(work, value_column, group_column="patient"):
    grouped=work.groupby(group_column)[value_column]
    counts=grouped.size().astype(float)
    means=grouped.mean()
    n_total=float(counts.sum()); n_groups=float(len(counts))
    if n_groups < 2 or n_total <= n_groups:
        raise RuntimeError("Insufficient replicated groups for variance decomposition")
    grand=float(work[value_column].mean())
    ss_between=float((counts*(means-grand)**2).sum())
    centered=work[value_column]-work[group_column].map(means)
    ss_within=float((centered**2).sum())
    ms_between=ss_between/(n_groups-1)
    ms_within=ss_within/(n_total-n_groups)
    effective_n=(n_total-float((counts**2).sum())/n_total)/(n_groups-1)
    between=max((ms_between-ms_within)/effective_n,0.0)
    within=max(ms_within,0.0)
    return between,within

def variance_components(work, value_column):
    errors=[]
    for optimizer in ["lbfgs","powell"]:
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                fit=MixedLM.from_formula(
                    f"{value_column} ~ 1",groups="patient",data=work
                ).fit(reml=True,method=optimizer,disp=False)
            between=float(fit.cov_re.iloc[0,0]); within=float(fit.scale)
            if np.isfinite([between,within]).all() and between >= 0 and within > 0:
                return between,within,f"MixedLM REML ({optimizer})"
        except Exception as error:
            errors.append(f"{optimizer}: {type(error).__name__}: {error}")
    between,within=method_of_moments_variance(work,value_column)
    warnings.warn(
        f"MixedLM failed for {value_column}; using one-way random-effects "
        f"method of moments. Attempts: {'; '.join(errors)}"
    )
    return between,within,"One-way random-effects method of moments"

rows=[]
for gene in GENES:
    column=f"z_{gene}"; work=regions[["patient",column]].dropna().copy()
    between,within,variance_method=variance_components(work,column)
    total=between+within
    if not np.isfinite(total) or total <= 0:
        raise RuntimeError(f"Invalid variance components for {gene}")
    icc=between/total
    per=work.groupby("patient")[column].agg(
        n_regions="size",regional_range=lambda x:float(x.max()-x.min()),regional_sd="std"
    )
    multi=per.loc[per.n_regions.ge(2)]
    rows.append({"gene":gene,"n_regions":len(work),"n_patients":work.patient.nunique(),"n_multiregion":len(multi),
                 "between_variance":between,"within_variance":within,"icc":icc,
                 "median_within_range":multi.regional_range.median(),"median_within_sd":multi.regional_sd.median(),
                 "variance_method":variance_method})
panel_a=pd.DataFrame(rows)
panel_a.to_csv(OUTPUT_DIR/"panel_a_gene_regional_variation.csv",index=False)
display(panel_a)


In [ ]:
# S4B: recover both public evolutionary classifications at patient level.
evo_col=find_column(primary_annotation.columns,["evo","type"])
evo=primary_annotation[[patient_column,evo_col]].groupby(patient_column,as_index=False).first().rename(columns={patient_column:"patient",evo_col:"EvoType"})
omnibus_data=patients.merge(evo,on="patient",how="left",validate="one_to_one")

def categorical_model(data,column,label):
    fields=["mitoxy",column,"grade","stage","age","purity","wgi","ith"]
    complete=data[fields].replace([np.inf,-np.inf],np.nan).dropna().copy()
    levels=sorted(complete[column].astype(str).unique()); reference=levels[0]
    dummies=pd.get_dummies(complete[column].astype(str),prefix=column,drop_first=True,dtype=float)
    X=pd.concat([dummies,complete[["grade","stage","age","purity","wgi","ith"]].astype(float)],axis=1)
    X=sm.add_constant(X,has_constant="add"); fit=sm.OLS(complete.mitoxy.astype(float),X).fit(cov_type="HC3",use_t=False)
    terms=list(dummies.columns); restriction=np.zeros((len(terms),len(fit.params)))
    for i,term in enumerate(terms): restriction[i,list(fit.params.index).index(term)]=1
    test=fit.wald_test(restriction,scalar=True)
    rows=[]
    for term in terms:
        ci=fit.conf_int().loc[term]
        rows.append({"classification":label,"reference":reference,"level":term.replace(column+"_","",1),"n":len(complete),
                     "effect":fit.params[term],"se":fit.bse[term],"ci_low":ci.iloc[0],"ci_high":ci.iloc[1],"p_value":fit.pvalues[term],
                     "omnibus_df":len(terms),"omnibus_p":float(test.pvalue),"r_squared":fit.rsquared})
    return rows

panel_b=pd.DataFrame(categorical_model(omnibus_data,"condition","Evolutionary condition")+
                     categorical_model(omnibus_data,"EvoType","Evolutionary subtype"))
panel_b.to_csv(OUTPUT_DIR/"panel_b_evolution_omnibus.csv",index=False); display(panel_b)


In [ ]:
contracts = {
    'panel_a_gene_regional_variation.csv': ['gene', 'n_regions', 'n_patients', 'icc', 'median_within_range', 'variance_method'],
    'panel_b_evolution_omnibus.csv': ['classification', 'reference', 'level', 'n', 'effect', 'ci_low', 'ci_high', 'p_value', 'omnibus_p'],
}
rows=[]
for filename,required in contracts.items():
    path=OUTPUT_DIR/filename
    table=pd.read_csv(path)
    missing=sorted(set(required)-set(table.columns))
    if missing: raise KeyError(f"{filename} missing {missing}")
    rows.append({"file":filename,"rows":len(table),"size_bytes":path.stat().st_size,"sha256":sha256_file(path)})
processed_manifest=pd.DataFrame(rows)
processed_manifest.to_csv(OUTPUT_DIR/"processed_manifest.csv",index=False)
display(processed_manifest)
print("Processed-file handoff: PASS")
